In [7]:
# ============================================================
# PySpark UDFs — Memory, Performance, and Best Practices
# ============================================================

# 1. Python vs JVM Architecture
# - Spark runs on JVM (Java Virtual Machine)
# - PySpark code runs in separate Python worker processes
# - Communication happens via serialization (JVM ↔ Python)
# - Memory is NOT shared between JVM and Python

# Implication:
# Python workers can consume additional memory outside JVM limits
# This can lead to high memory usage or OutOfMemory (OOM) errors


# 2. Why Python UDFs Are Expensive
# - Data is copied from JVM → Python → JVM
# - Standard UDF processes data row-by-row
# - Serialization/deserialization overhead
# - Python memory is less efficiently managed (GC behavior)

# Result:
# - Slower execution
# - Higher memory consumption
# - Possible OOM errors in large datasets


# 3. Best Practice Order (VERY IMPORTANT)

# 1. Use Built-in Spark Functions (BEST)
# - Runs entirely in JVM
# - No serialization
# - Fastest and most memory-efficient

# Example:
# from pyspark.sql.functions import upper
# df.withColumn("name_upper", upper(df["name"]))


# 2. Use Higher-Order Functions (for arrays/maps)
# - Still JVM-based
# - No Python overhead

# Example:
# from pyspark.sql.functions import expr
# df.select(expr("transform(arr, x -> x + 1)"))


# 3. Use Pandas UDF (Vectorized UDF)
# - Uses Apache Arrow for fast data transfer
# - Processes data in batches instead of row-by-row
# - Much faster than standard Python UDF

# Example:
# from pyspark.sql.functions import pandas_udf
# import pandas as pd
#
# @pandas_udf("string")
# def to_upper_pandas(s: pd.Series) -> pd.Series:
#     return s.str.upper()


# 4. Use Scala/Java UDF (if needed)
# - Runs inside JVM
# - No Python serialization overhead
# - Better for heavy logic and large data


# 5. Avoid Standard Python UDF (LAST RESORT)
# - Row-by-row execution
# - High serialization cost
# - High memory usage
# - Slow performance


# 4. Key Insight
# The main issue is NOT just Python vs JVM separation,
# but the cost of data movement and duplication.

# Each UDF call:
# - Copies data JVM → Python
# - Processes it
# - Sends it back to JVM

# Expensive in:
# - CPU
# - Memory
# - Network (in distributed systems)


# 5. When to Use Scala/Java UDF
# - Complex logic that cannot be expressed using built-in functions
# - Large-scale data processing
# - Performance or memory issues with Python UDFs


# 6. Rule of Thumb
# Before writing a UDF, always ask:
# 1. Can I use built-in Spark functions?
# 2. Can I use higher-order functions?
# 3. Can I use Pandas UDF?
# 4. Only then use Python UDF


# Final Takeaway
# - Python runs separately → can increase memory usage
# - UDFs can cause OOM issues

In [8]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("User Defined Functions")
    .master("local[*]")
    .config("spark.executor.cores", 2)
    .config("spark.cores.max", 6)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark

In [13]:
# Read employee data

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

emp = spark.read.format("csv").option("header", True).schema(emp_schema).load("data/input/emp.csv")

emp.rdd.getNumPartitions()

emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [14]:
# Create a function to generate 10% of Salary as Bonus

def bonus(salary):
    return int(salary) * 0.1

In [15]:
# Register as UDF
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

bonus_udf = udf(bonus, DoubleType())

# Register as SQL UDF
spark.udf.register("bonus_sql_udf", bonus, "double")

<function __main__.bonus(salary)>

In [23]:
emp.withColumn("bonus", bonus_udf(emp["salary"])).show()

df = emp.select("salary", bonus_udf(emp["salary"]).alias("bonus"))

# Using Multiple Columns
df.withColumn("result", bonus_udf(df["salary"], df["bonus"]))

+-----------+-------------+-------------+---+------+------+----------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+------+----------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|5000.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|4500.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|5500.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|4800.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|6000.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|5200.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|7000.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|5100.0|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|5800.0|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-

DataFrame[salary: string, bonus: double, result: double]

In [17]:
# Create new column as bonus using UDF
from pyspark.sql.functions import expr

emp.withColumn("bonus", expr("bonus_sql_udf(salary)")).show()

+-----------+-------------+-------------+---+------+------+----------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+------+----------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|5000.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|4500.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|5500.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|4800.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|6000.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|5200.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|7000.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|5100.0|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|5800.0|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-

In [18]:
# Create new column as bonus without UDF

emp.withColumn("bonus", expr("salary * 0.1")).show()

+-----------+-------------+-------------+---+------+------+----------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+------+----------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|5000.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|4500.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|5500.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|4800.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|6000.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|5200.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|7000.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|5100.0|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|5800.0|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-

In [7]:
spark.stop()